# ECC/EMC — Kaggle Training: EmoSet Semi-Supervised (SSL Only — Single Run)

Semi-supervised FixMatch + EMC-adaptive threshold on EmoSet-118K.

## Before you run
1. **Enable GPU:** Settings → Session options → GPU (P100 or T4). Phone verification required.
2. **Add datasets:** Public EmoSet-118K dataset in the sidebar.
3. **GitHub token secret:** `GITHUB_TOKEN` in Kaggle Settings → Secrets.
4. **Edit the config cell below** — set `LABEL_NUM` and `RULE` for this run.

## One run per session
Each SSL run takes ~6–8 hours on P100. One run per Kaggle session (12-hour cap).
Edit `LABEL_NUM` and `RULE` below, then **Save and Run All**.

## Results
Outputs go to `/kaggle/working/runs/emoset_ssl_n<labels>_<rule>/`.
A zip cell at the end creates `emoset_ssl_results.zip` for one-click download.


In [ ]:
# ===================== CONFIGURE THESE — EDIT FOR EACH RUN =====================
# Labeled samples: 4000 or 8000 (EmoSet)
LABEL_NUM = 4000   # <-- CHANGE THIS for each run

# Rule: 'ratio' = EMC-adaptive threshold, 'fixed' = FixMatch baseline
RULE = 'ratio'   # <-- CHANGE THIS for each run ('ratio' or 'fixed')

# Public EmoSet-118K dataset
EMOSET_DATASET_SLUG = "georgeliil/emoset"   # <-- REPLACE if different

# ==========================================================

print(f"Label num: {LABEL_NUM}")
print(f"Rule: {RULE}")
print("EmoSet dataset slug:", EMOSET_DATASET_SLUG)

In [ ]:
# Install dependencies
!pip -q install torch torchvision numpy pyyaml pillow

In [ ]:
# Clone repo using GitHub token secret (added in Kaggle Settings → Secrets)
import os
import subprocess

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    secret_token = user_secrets.get_secret("GITHUB_TOKEN")
    repo_url = f"https://{secret_token}@github.com/Rahul-Sai-Indeevar/To-Err-Like-Human.git"
except Exception:
    repo_url = "https://github.com/Rahul-Sai-Indeevar/To-Err-Like-Human.git"

dest = "/kaggle/working/ecc-ver"

if not os.path.exists(dest):
    subprocess.run(["git", "clone", repo_url, dest], check=True)
    print("Cloned repo to", dest)
else:
    print("Repository already exists at", dest)

os.chdir(dest)
print("Current dir:", os.getcwd())

In [ ]:
# Install ecc-ver package
!pip -q install -e .

In [ ]:
# Symlink EmoSet dataset into data/ layout
import os
from pathlib import Path

os.makedirs("data", exist_ok=True)

# Robustly find EmoSet/ anywhere under /kaggle/input
# Kaggle mounts private datasets under /kaggle/input/datasets/<user>/<slug>/...
found_images = False
found_splits = False
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth > 5:
        dirs[:] = []
    for d in dirs:
        if d in ("image", "EmoSet") and not found_images:
            src = os.path.join(root, d)
            if any((Path(src) / c).is_dir() for c in ["amusement", "anger", "awe", "contentment", "disgust", "excitement", "fear", "sadness"]):
                dst = "data/EmoSet"
                if not os.path.exists(dst):
                    os.symlink(src, dst)
                    print(f"Linked EmoSet images -> data/EmoSet (from {src})")
                found_images = True
        if d == "EmoSet_splits" and not found_splits:
            src = os.path.join(root, d)
            dst = "data/EmoSet_splits"
            if not os.path.exists(dst):
                os.symlink(src, dst)
                print(f"Linked EmoSet splits -> data/EmoSet_splits (from {src})")
            found_splits = True

if not found_images:
    print("WARNING: Could not find EmoSet image directory under /kaggle/input")
if not found_splits:
    print("WARNING: Could not find EmoSet_splits under /kaggle/input")

print("\nData layout:")
!find data -maxdepth 3 | head -50

In [ ]:
# Quick smoke test
!python -m pytest -q

In [ ]:
# ===================== SEMI-SUPERVISED (EmoSet, Section 4.2.1) =====================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"=== SSL n={LABEL_NUM}, rule={RULE} ===")
cmd = f"python -m ecc.ssl --config configs/ssl_fi.yaml --set ssl.n_labeled={LABEL_NUM} ssl.rule={RULE} --set data.root=data/EmoSet data.splits_dir=data/EmoSet_splits output_dir=/kaggle/working/runs/emoset_ssl_n{LABEL_NUM}_{RULE}"
print("Running:", cmd)
!{cmd}

print("SSL done.")

In [ ]:
# ===================== PREPARE RESULTS ZIP FOR ONE-CLICK DOWNLOAD =====================
import zipfile, os, glob, shutil

run_dir = f"/kaggle/working/runs/emoset_ssl_n{LABEL_NUM}_{RULE}"
zip_path = f"/kaggle/working/emoset_ssl_n{LABEL_NUM}_{RULE}_results.zip"

if not os.path.exists(run_dir):
    print(f"No run directory found: {run_dir}")
else:
    files_to_zip = ["best.pt", "checkpoint.pt", "results.json", "history.jsonl", "config.json"]
    
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in files_to_zip:
            fpath = os.path.join(run_dir, fname)
            if os.path.exists(fpath):
                zf.write(fpath, arcname=fname)
                print(f"Added {fname}")
            else:
                print(f"Skipped {fname} (not found)")
    
    print(f"\nCreated {zip_path}")
    print("Contents:")
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            print(f"  {info.filename} ({info.file_size:,} bytes)")
    
    print("\nDownload from Output panel (right sidebar → Output).")

## Resuming after session timeout

Kaggle free sessions are capped at ~12 hours. Every epoch saves `checkpoint.pt` to:
1. The run's `output_dir` (`/kaggle/working/runs/emoset_ssl_n<labels>_<rule>/`), and
2. `PERSISTENT_DIR` (your mounted Kaggle Dataset, if set — see notebook 1).

If a session dies, re-run the same cell — it auto-detects `checkpoint.pt` in the output dir
**and** in `PERSISTENT_DIR`, resuming from the last epoch.

To force a specific checkpoint, add `--set resume_from=/kaggle/working/runs/emoset_ssl_n<labels>_<rule>/checkpoint.pt`.

## Download results

Run the zip cell above to create `emoset_ssl_n<labels>_<rule>_results.zip` containing:
- `best.pt` — best-on-val model weights
- `checkpoint.pt` — resumable training checkpoint
- `results.json` — test metrics (ACC, ECC, EMC, ACC2)
- `history.jsonl` — per-epoch training log
- `config.json` — exact config used

Download the zip from the notebook output panel before the session ends.

## Running all 4 SSL experiments

Run each as a separate Kaggle session:

| Session | LABEL_NUM | RULE  | Expected time |
|---------|-----------|-------|---------------|
| 1       | 4000      | ratio | ~6–8 hours    |
| 2       | 4000      | fixed | ~6–8 hours    |
| 3       | 8000      | ratio | ~6–8 hours    |
| 4       | 8000      | fixed | ~6–8 hours    |

Between sessions:
1. Download `emoset_ssl_n<labels>_<rule>_results.zip` from the Output panel
2. Change `LABEL_NUM` and `RULE` at the top of this notebook
3. **Save and Run All** again
